# Analysis of uoe5 pipeline run
Reran antibody panel 1 with new updates within the pipeline, resutling in run uoe5.

Major pipeline changes include more quality control mesasures focused on organelle channels (lysosome, endosome, and mitochondria) - I added the contrast check that we run for the nuclues channel to script 3 to check each channel before segmentation is preformed. The organelle channels are generally more dim than the nuclues channel, so the threshold for QC is quite a bit lower than that of the nucleus channel.

The goals here are to preform the prior PCA analyses (plus additional analysis on the genotype level, manual focused analsysis, and structure level analysis) and assess whether we reduced the number of poor quality images that make it to this analysis stage. 

In [1]:
import pathlib as Path
import pandas as pd
import os

import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load metadata

In [2]:
METADATA_FN = "/data/kelpschdj/iNDI/Production/notebooks/COMBINED_ASP_BATCH_METADATA.csv"
METADATA = pd.read_csv(METADATA_FN)

print(METADATA.head())

  iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info     Gene  \
0    INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
1    INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
2    INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN  HNRNPA1   
3    INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
4    INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN     PFN1   

  Variant Genotype          Gene_Var_Type  \
0   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
2   D262N   REV/WT   HNRNPA1_D262N_REV/WT   
3   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
4    C71G   SNV/WT       PFN1_C71G_SNV/WT   

                                      Cell_Line_Info     Source Asp PosID  \
0  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  automated        C4   
1  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  automated        C4   
2  JAX_ID_JIPSC001042\n

# Organelle feature data
Pull in organelle feature data (Lysosome, Endosome, Mitochondria)

In [3]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20260901_155524_uoe5/organelle_features/*_organelle_features_*.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())

028ebee9-afaf-4ff8-b435-af11714285dc_organelle_features__20260902_114905.parquet: 116923 rows
07fc5da6-9d7d-4c97-858b-4b76df1859a5_organelle_features__20260902_114905.parquet: 115713 rows
2350e82d-40dd-4bb4-93de-44edec0d1718_organelle_features__20260902_114905.parquet: 138260 rows
2b1c31ad-579d-4ea6-a782-251ea083ed6a_organelle_features__20260902_114905.parquet: 206382 rows
2da14033-ab86-4f33-b4f7-ef375c80ce62_organelle_features__20260902_114905.parquet: 166866 rows
32d75d22-bd17-4629-8257-a4056e3ffcee_organelle_features__20260902_114905.parquet: 107005 rows
4405a3b2-6b88-49b1-91f3-992e09ccbd16_organelle_features__20260902_114905.parquet: 131590 rows
51cf625a-9422-406f-aa9e-a9f1238d1b96_organelle_features__20260902_114905.parquet: 177582 rows
72c0a713-b5d5-4cd4-a826-c556368ed1d1_organelle_features__20260902_114905.parquet: 100645 rows
7e03b502-ad64-4090-a9e2-fc4c3ecdae11_organelle_features__20260902_114905.parquet: 169399 rows
934f6de5-f58b-4a1e-b225-ac6414500bb7_organelle_features__202

## Merge with metadata 

In [4]:
print(df.columns.tolist())
print(METADATA.columns.tolist())

print(METADATA["Well_ID"].head())
print(df[["Row", "Column"]].head())

df["Well_ID"] = (
    df["Row"].apply(lambda r: chr(ord("A") + int(r) - 1))
    + df["Column"].astype(int).astype(str).str.zfill(2)
)

df = df.merge(
    METADATA,
    left_on=["Measurement_ID", "Well_ID"],
    right_on=["Folder_ID", "Well_ID"],
    how="left",
)

['Measurement_ID', 'subdirectory', 'Row', 'Column', 'Frame', 'Plane', 'Time', 'DAPI_filename', 'channel_filename', 'Structure', 'Stain', 'Nucleus_ID', 'frame_std', 'contrast_check', 'area_px', 'organelle_count', 'average_organelle_area', 'roi_occupancy', 'max_f_intensity', 'sum_f_intensity', 'mean_f_intensity', 'median_f_intensity', 'CoefOfVar_intensity', 'area_mean', 'area_median', 'area_std', 'area_p10', 'area_p90', 'perimeter_mean', 'perimeter_median', 'perimeter_std', 'perimeter_p10', 'perimeter_p90', 'eccentricity_mean', 'eccentricity_median', 'eccentricity_std', 'eccentricity_p10', 'eccentricity_p90', 'solidity_mean', 'solidity_median', 'solidity_std', 'solidity_p10', 'solidity_p90', 'extent_mean', 'extent_median', 'extent_std', 'extent_p10', 'extent_p90', 'axis_major_length_mean', 'axis_major_length_median', 'axis_major_length_std', 'axis_major_length_p10', 'axis_major_length_p90', 'axis_minor_length_mean', 'axis_minor_length_median', 'axis_minor_length_std', 'axis_minor_length_

## Well-level summary

In [5]:
for plate, grp in df.groupby('iNDI_Plate_ID'):
    print(f"\n{plate}")
    print(grp.groupby('Gene_Var_Type')['Well_ID'].nunique())


INDI00002D
Gene_Var_Type
FUS_R495X_REV/WT     48
FUS_R495X_SNV/SNV    48
FUS_R495X_SNV/WT     48
KOLF2.1J             96
VCP_R159H_REV/WT     48
VCP_R159H_SNV/SNV    48
VCP_R159H_SNV/WT     48
Name: Well_ID, dtype: int64

INDI00007D
Gene_Var_Type
KOLF2.1J                96
TARDBP_M337V_REV/WT     48
TARDBP_M337V_SNV/WT     48
TARDBP_Q331K_SNV/SNV    47
TIA1_P351L_REV/WT       48
TIA1_P351L_SNV/SNV      48
Name: Well_ID, dtype: int64

INDI00011D
Gene_Var_Type
FUS_R216C_SNV/SNV      47
KOLF2.1J               96
PSEN2_N141I_REV/WT     47
PSEN2_N141I_SNV/SNV    48
TBK1_E696K_REV/WT       3
TBK1_E696K_SNV/SNV      4
TBK1_E696K_SNV/WT       2
Name: Well_ID, dtype: int64

INDI00015D
Gene_Var_Type
APP_A673T_REV/WT      48
APP_A673T_SNV/WT      48
APP_E693G_SNV/SNV     48
APP_E693G_SNV/WT      48
APP_V717I_SNV/SNV     48
KOLF2.1J              48
PSEN1_E276A_SNV/WT    48
Name: Well_ID, dtype: int64

INDI00019D
Gene_Var_Type
APP_E693G_REV/WT       41
FUS_R216C_REV/WT       47
FUS_R216C_SNV/WT   

## Clean-up data
If a nuclear ROI does not have segmentation for all three channels, throw it away

In [6]:
## add new analysis to check number of channels per nucleus is 4 (including nuclear channel) for all nuclei in the dataset - If a nucleus has less than 4 channels remove it from analysis
counts = (
    df.groupby(["DAPI_filename", "Nucleus_ID"])["Structure"]
        .nunique()
)

keep_keys = counts[counts == 3].index


filtered = df.set_index(["DAPI_filename", "Nucleus_ID"]).loc[keep_keys].reset_index()
print(f"Filtered dataset: {len(filtered)} rows (from {len(df)} original rows)")

Filtered dataset: 3121178 rows (from 3125326 original rows)


Reduced by ~ 4500 ROIs

In [7]:
df = filtered  # replace the original df with the filtered one for further analysis

## Apply subjective filters from Marianita

In [8]:
# MS_filter 
MS_filter_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel1/marianita_excluded_genotypes_by_plate.csv"
MS_filter = pd.read_csv(MS_filter_fn)
print("MS_filter columns:", MS_filter.columns.tolist())
print(MS_filter.head())

def norm_geno(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace("_", " ", regex=False)   # underscores -> spaces
         .str.replace(r"\s+", " ", regex=True)  # collapse repeats
         .str.upper()                            # case-insensitive
    )

remove_pairs = set(zip(
    MS_filter["iNDI_Plate_ID"].astype(str).str.strip(),
    norm_geno(MS_filter["Genotype_Name"]),
))

key = list(zip(
    df["iNDI_Plate_ID"].astype(str).str.strip(),
    norm_geno(df["Genotype_Name"]),
))
mask = pd.Series(key, index=df.index).isin(remove_pairs)

filtered = df[~mask].copy()
print(f"removed {mask.sum()} rows; {len(filtered)} remain")

# confirm the normalization actually bridged the two formats
missing = remove_pairs - set(key)
if missing:
    print(f"note: {len(missing)} pair(s) still unmatched after normalization:")
    for p in sorted(missing):
        print("  ", p)

MS_filter columns: ['iNDI_Plate_ID', 'Genotype_Name']
  iNDI_Plate_ID    Genotype_Name
0    INDI00007D   TIA1_P351L_HET
1    INDI00008D   TIA1_P351L_HET
2    INDI00011D  PSEN2_N141I_REV
3    INDI00011D   TBK1_E696K_HET
4    INDI00011D   TBK1_E696K_HOM
removed 22895 rows; 3098283 remain
note: 27 pair(s) still unmatched after normalization:
   ('INDI00007D', 'TIA1 P351L HET')
   ('INDI00008D', 'TIA1 P351L HET')
   ('INDI00016D', 'PSEN1 E276A HET')
   ('INDI00020D', 'APP E693G REV')
   ('INDI00020D', 'PSEN1 E276A HOM')
   ('INDI00020D', 'PSEN2 N141I HET')
   ('INDI00023D', 'SNCB P123H REV')
   ('INDI00024D', 'SNCB P123H HET')
   ('INDI00024D', 'SNCB P123H HOM')
   ('INDI00024D', 'SNCB P123H REV')
   ('INDI00026D', 'HNRNPA1 D262N REV')
   ('INDI00026D', 'PFN1 C71G HET')
   ('INDI00028D', 'HNRNPA1 D262N HOM')
   ('INDI00028D', 'HNRNPA1 D262N REV')
   ('INDI00028D', 'PFN1 C71G HET')
   ('INDI00039D', 'SNCB P123H HET')
   ('INDI00040D', 'PSEN2 N141I HET')
   ('INDI00040D', 'SNCB P123H HET')
 

In [9]:
print(filtered.head())
print(filtered.shape)
# pull unique DAPI_filename, Measurment_ID, Nucleus_ID

filter_key = filtered[["DAPI_filename", "Measurement_ID", "Nucleus_ID", "JAX_ID"]].drop_duplicates().reset_index(drop=True)
filter_key = filter_key.rename(columns = {"DAPI_filename": "filename", "Nucleus_ID": "label"})
print(filter_key.head())
print(filter_key.shape)

                DAPI_filename  Nucleus_ID  \
0   r01c01f01p01-ch01t01.tiff         1.0   
1   r01c01f01p01-ch01t01.tiff         1.0   
2   r01c01f01p01-ch01t01.tiff         1.0   
9   r01c01f01p01-ch01t01.tiff         2.0   
10  r01c01f01p01-ch01t01.tiff         2.0   

                          Measurement_ID subdirectory  Row  Column  Frame  \
0   7e03b502-ad64-4090-a9e2-fc4c3ecdae11       r01c01    1       1      1   
1   7e03b502-ad64-4090-a9e2-fc4c3ecdae11       r01c01    1       1      1   
2   7e03b502-ad64-4090-a9e2-fc4c3ecdae11       r01c01    1       1      1   
9   72c0a713-b5d5-4cd4-a826-c556368ed1d1       r01c01    1       1      1   
10  72c0a713-b5d5-4cd4-a826-c556368ed1d1       r01c01    1       1      1   

    Plane  Time           channel_filename  ... Variant Genotype  \
0       1     1  r01c01f01p01-ch04t01.tiff  ...     NaN      NaN   
1       1     1  r01c01f01p01-ch02t01.tiff  ...     NaN      NaN   
2       1     1  r01c01f01p01-ch03t01.tiff  ...     NaN      N

## Write final output

In [10]:
filtered.to_parquet("uoe5_organelle_output.parquet", index=False)

# Nucleus feature data

## Merge with metadata

In [11]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20260901_155524_uoe5/nuclei_filtered/*_nuclei_filtered.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())
print(df.shape)

df["Well_ID"] = (
    df["Row"].apply(lambda r: chr(ord("A") + int(r) - 1))
    + df["Column"].astype(int).astype(str).str.zfill(2)
)

df = df.merge(
    METADATA,
    left_on=["Measurement_ID", "Well_ID"],
    right_on=["Folder_ID", "Well_ID"],
    how="left",
)

print(df.head())
print(df.shape)

028ebee9-afaf-4ff8-b435-af11714285dc_nuclei_filtered.parquet: 87928 rows
07fc5da6-9d7d-4c97-858b-4b76df1859a5_nuclei_filtered.parquet: 81886 rows
2350e82d-40dd-4bb4-93de-44edec0d1718_nuclei_filtered.parquet: 98782 rows
2b1c31ad-579d-4ea6-a782-251ea083ed6a_nuclei_filtered.parquet: 124127 rows
2da14033-ab86-4f33-b4f7-ef375c80ce62_nuclei_filtered.parquet: 109822 rows
32d75d22-bd17-4629-8257-a4056e3ffcee_nuclei_filtered.parquet: 68940 rows
4405a3b2-6b88-49b1-91f3-992e09ccbd16_nuclei_filtered.parquet: 90673 rows
51cf625a-9422-406f-aa9e-a9f1238d1b96_nuclei_filtered.parquet: 125952 rows
72c0a713-b5d5-4cd4-a826-c556368ed1d1_nuclei_filtered.parquet: 70820 rows
7e03b502-ad64-4090-a9e2-fc4c3ecdae11_nuclei_filtered.parquet: 112023 rows
934f6de5-f58b-4a1e-b225-ac6414500bb7_nuclei_filtered.parquet: 110118 rows
9697941f-11d4-43bb-8f50-d424c4e05c29_nuclei_filtered.parquet: 109010 rows
b96883e9-18b4-4f0d-ba6e-0ddd08c5413f_nuclei_filtered.parquet: 102498 rows
c170959e-633a-458c-8542-2aa619c0d7f8_nuclei_

## QC filter

In [12]:
keys = ["filename", "Measurement_ID", "label", "JAX_ID"]
subset = df[df.set_index(keys).index.isin(filter_key.set_index(keys).index)]
print(subset.shape)

(1036846, 57)


## Write final output

In [13]:
subset.to_parquet("uoe5_nucleus_output.parquet", index=False)